In [1]:
# import
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import re
import os
from pathlib import Path
import glob
from brainglobe_atlasapi import BrainGlobeAtlas

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

import seaborn as sns
import matplotlib.pyplot as plt
from scipy import sparse
from matplotlib.colors import Normalize

In [ ]:
############ Whole Sample ##############

In [ ]:
# read data
merged_data = pd.read_csv("PB_WholeBrain_All_QCed.txt",sep="\t")

In [ ]:
# DE analysis function
def run_deseq2(counts, metadata):

    counts_sub = counts.copy()
    meta_sub = metadata.copy()

    dds = DeseqDataSet(
        counts=counts_sub,
        metadata=meta_sub,
        design_factors=["genotype", "month"]
    )

    # normalization + dispersion + model fitting
    dds.deseq2()

    stat_res = DeseqStats(
        dds,
        contrast=[
            "genotype",
            "5xFAD",
            "WT"
        ]
    )

    stat_res.summary()

    res = stat_res.results_df
    
    res["compare"] = "whole_brain"
    res["type"] = "whole_brain"
    res["gene"] = res.index

    return res

genes = ["Eif4g3", "Ece1", "Hp1bp3", "Kif17"]

In [ ]:
######### forebrain
tmp_dat = merged_data[merged_data["brain"] == "forebrain"].copy()

## create metadata and count
meta_data = tmp_dat[["SampleID","genotype", "month", "n_cells", "brain"]]
count_data = tmp_dat.drop(
    columns=["SampleID","genotype", "month", "n_cells", "brain"]
)

In [ ]:
whole_forebrain = run_deseq2(
    count_data,
    meta_data
)
whole_forebrain.loc[genes]

In [ ]:
######### hindbrain
tmp_dat = merged_data[merged_data["brain"] == "hindbrain"].copy()

## create metadata and count
meta_data = tmp_dat[["SampleID","genotype", "month", "n_cells", "brain"]]
count_data = tmp_dat.drop(
    columns=["SampleID","genotype", "month", "n_cells", "brain"]
)

In [ ]:
whole_hindbrain = run_deseq2(
    count_data,
    meta_data
)
whole_hindbrain.loc[genes]

In [ ]:
############ Whole HPF ##############

In [ ]:
# read data
merged_data = pd.read_csv("PB_HindbrainRegion_All_QCed.txt",sep="\t")

In [ ]:
# DE analysis function
def run_deseq2(counts, metadata):

    counts_sub = counts.copy()
    meta_sub = metadata.copy()

    dds = DeseqDataSet(
        counts=counts_sub,
        metadata=meta_sub,
        design_factors=["genotype", "month"]
    )

    # normalization + dispersion + model fitting
    dds.deseq2()

    stat_res = DeseqStats(
        dds,
        contrast=[
            "genotype",
            "5xFAD",
            "WT"
        ]
    )

    stat_res.summary()

    res = stat_res.results_df
    
    res["compare"] = "whole_hpf"
    res["type"] = "whole_hpf"
    res["gene"] = res.index

    return res

genes = ["Eif4g3", "Ece1", "Hp1bp3", "Kif17"]

In [ ]:
######### HPF
tmp_dat = merged_data[merged_data["region"] == "HPF"].copy()

## create metadata and count
meta_data = tmp_dat[["SampleID","genotype", "month", "region", "n_cells", "brain"]]
count_data = tmp_dat.drop(
    columns=["SampleID","genotype", "month", "region", "n_cells", "brain"]
)

res_HPF = run_deseq2(
    count_data,
    meta_data
)
res_HPF.loc[genes]

In [ ]:
######### Others
tmp_dat = merged_data[merged_data["region"] == "Others"].copy()

## create metadata and count
meta_data = tmp_dat[["SampleID","genotype", "month", "region", "n_cells", "brain"]]
count_data = tmp_dat.drop(
    columns=["SampleID","genotype", "month", "region", "n_cells", "brain"]
)

res_Others = run_deseq2(
    count_data,
    meta_data
)
res_Others.loc[genes]

In [ ]:
############ HPF/Others Cell type ##############

In [ ]:
# read data
merged_data = pd.read_csv("PB_HPFOthersCellTypes_All_QCed.txt",sep="\t")

In [ ]:
####### stratified by cell type
def run_deseq2_by_celltype(counts, metadata, ct):
    
    idx = metadata["main_type"] == ct
    
    counts_sub = counts.loc[idx].copy()
    meta_sub = metadata.loc[idx].copy()
    
    if len(meta_sub) < 2:
        print(f"Warning: Celltype {ct} has only {len(meta_sub)} samples, skipping...")
        return None
    
    if len(meta_sub[meta_sub["genotype"] == "5xFAD"]) == 0 or \
       len(meta_sub[meta_sub["genotype"] == "WT"]) == 0:
        print(f"Warning: Celltype {ct} missing one genotype, skipping...")
        return None
    
    dds = DeseqDataSet(
        counts=counts_sub,
        metadata=meta_sub,
        design_factors=["genotype","month"]
    )
    
    dds.deseq2()
    
    stat_res = DeseqStats(
        dds,
        contrast=[
            "genotype",
            "5xFAD",
            "WT"
        ]
    )
    
    stat_res.summary()
    res = stat_res.results_df.copy()
    
    res["compare"] = "HPF/Others cell types"
    res["type"] = ct
    res["gene"] = res.index
    
    return res

In [ ]:
######### HPF
tmp_dat = merged_data[merged_data["region"] == "HPF"].copy()
meta_data = tmp_dat[["SampleID", "genotype", "month", "main_type", "region", "n_cells", "brain"]]
count_data = tmp_dat.drop(
    columns=["SampleID", "genotype", "month", "main_type", "region", "n_cells", "brain"]
)

unique_ct = sorted(meta_data["main_type"].unique())
print(f"Processing ct: {unique_ct}")

all_results = []
for ct_val in unique_ct:
    print(f"\n=== Processing ct: {ct_val} ===")
    
    result = run_deseq2_by_celltype(
        counts=count_data,
        metadata=meta_data,
        ct=ct_val
    )
    if result is not None:
        all_results.append(result)
    else:
        print(f"Skipping celltype {ct_val}")

if all_results:
    HPF_byCellType = pd.concat(all_results, ignore_index=True)
    
    if "gene" in HPF_byCellType.columns:
        cols = ["gene", "type"] + [col for col in HPF_byCellType.columns 
                                   if col not in ["gene", "type"]]
        HPF_byCellType = HPF_byCellType[cols]
    
    print(f"\nTotal results: {len(HPF_byCellType)} rows")
    print(f"CellType included: {HPF_byCellType['type'].unique()}")
    
else:
    print("No results generated")
    HPF_byCellType = pd.DataFrame()

HPF_byCellType[HPF_byCellType['gene'].isin(genes)]

In [ ]:
######### Others
tmp_dat = merged_data[merged_data["region"] == "Others"].copy()

meta_data = tmp_dat[["SampleID", "genotype", "month", "main_type", "region", "n_cells", "brain"]]
count_data = tmp_dat.drop(
    columns=["SampleID", "genotype", "month", "main_type", "region", "n_cells", "brain"]
)

unique_ct = sorted(meta_data["main_type"].unique())
print(f"Processing ct: {unique_ct}")

all_results = []
for ct_val in unique_ct:
    print(f"\n=== Processing ct: {ct_val} ===")
    
    result = run_deseq2_by_celltype(
        counts=count_data,
        metadata=meta_data,
        ct=ct_val
    )
    if result is not None:
        all_results.append(result)
    else:
        print(f"Skipping celltype {ct_val}")

if all_results:
    Others_byCellType = pd.concat(all_results, ignore_index=True)
    
    if "gene" in Others_byCellType.columns:
        cols = ["gene", "type"] + [col for col in Others_byCellType.columns 
                                   if col not in ["gene", "type"]]
        Others_byCellType = Others_byCellType[cols]
    
    print(f"\nTotal results: {len(Others_byCellType)} rows")
    print(f"CellType included: {Others_byCellType['type'].unique()}")
    
else:
    print("No results generated")
    Others_byCellType = pd.DataFrame()

Others_byCellType[Others_byCellType['gene'].isin(["Eif4g3"])]

In [ ]:
DEres_combined = pd.concat([whole_forebrain, whole_hindbrain, res_HPF, res_Others, HPF_byCellType, Others_byCellType], axis=0)
DEres_combined.to_csv("AD_DEres_All.txt",index=False,sep="\t")